In [2]:
import duckdb

In [3]:
con = duckdb.connect("./data/reconciliation.duckdb")

In [4]:
con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT * FROM read_csv_auto('./data/orders.csv')
""")

In [5]:
con.execute("""
    CREATE OR REPLACE TABLE deliveries AS
    SELECT * FROM read_csv_auto('./data/deliveries.csv')
""")

In [6]:
con.sql("SELECT * FROM orders LIMIT 5").show()

┌──────────┬─────────────────┬────────┬────────────┐
│ order_id │    customer     │ amount │ order_date │
│ varchar  │     varchar     │ double │    date    │
├──────────┼─────────────────┼────────┼────────────┤
│ ORD00000 │ Patrick Sanchez │ 323.32 │ 2026-06-29 │
│ ORD00001 │ Megan Mcclain   │ 144.76 │ 2026-05-11 │
│ ORD00002 │ Allen Robinson  │ 370.87 │ 2026-06-24 │
│ ORD00003 │ Cristian Santos │ 447.17 │ 2026-06-25 │
│ ORD00004 │ Kevin Pacheco   │   24.6 │ 2026-06-07 │
└──────────┴─────────────────┴────────┴────────────┘



In [7]:
con.sql("SELECT * FROM deliveries LIMIT 5").show()

┌──────────┬──────────────────┬───────────────┐
│ order_id │ delivered_amount │ delivery_date │
│ varchar  │      double      │     date      │
├──────────┼──────────────────┼───────────────┤
│ ORD00001 │           144.76 │ 2026-05-19    │
│ ORD00002 │           370.87 │ 2026-07-18    │
│ ORD00003 │            425.5 │ 2026-07-21    │
│ ORD00004 │             24.6 │ 2026-06-07    │
│ ORD00006 │           107.43 │ 2026-06-18    │
└──────────┴──────────────────┴───────────────┘



In [8]:
missing_deliveries = con.sql("""
    SELECT o.*
    FROM orders o
    LEFT JOIN deliveries d ON o.order_id = d.order_id
    WHERE d.order_id IS NULL
""")

missing_deliveries.show()

┌──────────┬───────────────────┬────────┬────────────┐
│ order_id │     customer      │ amount │ order_date │
│ varchar  │      varchar      │ double │    date    │
├──────────┼───────────────────┼────────┼────────────┤
│ ORD00000 │ Patrick Sanchez   │ 323.32 │ 2026-06-29 │
│ ORD00005 │ Melissa Peterson  │ 257.62 │ 2026-07-10 │
│ ORD00019 │ David Hoffman     │ 355.24 │ 2026-05-11 │
│ ORD00032 │ Mr. Philip Cannon │ 122.23 │ 2026-07-08 │
│ ORD00043 │ Maria Brown       │  54.55 │ 2026-07-02 │
│ ORD00130 │ Sandra Sanchez    │ 330.45 │ 2026-05-05 │
│ ORD00140 │ John Kelly        │ 145.57 │ 2026-07-19 │
│ ORD00191 │ Jordan Morris     │ 278.49 │ 2026-07-11 │
│ ORD00201 │ Michael Mitchell  │ 107.57 │ 2026-05-04 │
│ ORD00244 │ Jerome Cox        │ 286.55 │ 2026-07-09 │
│    ·     │     ·             │    ·   │     ·      │
│    ·     │     ·             │    ·   │     ·      │
│    ·     │     ·             │    ·   │     ·      │
│ ORD00756 │ Tiffany Curtis    │ 397.28 │ 2026-05-23 │
│ ORD00781

In [9]:
amount_mismatches = con.sql("""
    SELECT 
        o.order_id, 
        o.amount AS order_amount, 
        d.delivered_amount,
        o.amount - d.delivered_amount AS difference
    FROM orders o
    JOIN deliveries d ON o.order_id = d.order_id
    WHERE o.amount != d.delivered_amount
""")

amount_mismatches.show()

┌──────────┬──────────────┬──────────────────┬────────────────────┐
│ order_id │ order_amount │ delivered_amount │     difference     │
│ varchar  │    double    │      double      │       double       │
├──────────┼──────────────┼──────────────────┼────────────────────┤
│ ORD00003 │       447.17 │            425.5 │ 21.670000000000016 │
│ ORD00012 │       174.93 │           169.19 │  5.740000000000009 │
│ ORD00064 │       160.68 │           116.66 │  44.02000000000001 │
│ ORD00065 │       474.01 │            449.2 │ 24.810000000000002 │
│ ORD00080 │       431.22 │           418.56 │ 12.660000000000025 │
│ ORD00085 │        103.3 │            81.18 │  22.11999999999999 │
│ ORD00088 │       492.24 │           471.51 │ 20.730000000000018 │
│ ORD00108 │       376.51 │           346.89 │ 29.620000000000005 │
│ ORD00139 │       424.59 │           402.83 │  21.75999999999999 │
│ ORD00144 │        14.74 │           -29.53 │              44.27 │
│    ·     │          ·   │              ·   │  

In [10]:
duplicate_deliveries = con.sql("""
    SELECT order_id, COUNT(*) AS delivery_count
    FROM deliveries
    GROUP BY order_id
    HAVING COUNT(*) > 1
""")

duplicate_deliveries.show()

┌──────────┬────────────────┐
│ order_id │ delivery_count │
│ varchar  │     int64      │
├──────────┼────────────────┤
│ ORD00256 │              2 │
│ ORD00312 │              2 │
│ ORD00321 │              2 │
│ ORD00161 │              2 │
│ ORD00372 │              2 │
│ ORD00683 │              2 │
│ ORD00739 │              2 │
│ ORD00754 │              2 │
│ ORD00930 │              2 │
│ ORD00962 │              2 │
│    ·     │              · │
│    ·     │              · │
│    ·     │              · │
│ ORD00208 │              2 │
│ ORD00267 │              2 │
│ ORD00274 │              2 │
│ ORD00587 │              2 │
│ ORD00898 │              2 │
│ ORD00067 │              2 │
│ ORD00269 │              2 │
│ ORD00534 │              2 │
│ ORD00113 │              2 │
│ ORD00151 │              2 │
└──────────┴────────────────┘
  36 rows         2 columns
  (20 shown)                



In [11]:
import sys
sys.path.append("./python")

from reconciliation import find_missing, find_amount_discrepancies, find_duplicates

In [12]:
find_missing(con, "orders", "deliveries", "order_id").show()

┌──────────┬───────────────────┬────────┬────────────┐
│ order_id │     customer      │ amount │ order_date │
│ varchar  │      varchar      │ double │    date    │
├──────────┼───────────────────┼────────┼────────────┤
│ ORD00000 │ Patrick Sanchez   │ 323.32 │ 2026-06-29 │
│ ORD00005 │ Melissa Peterson  │ 257.62 │ 2026-07-10 │
│ ORD00019 │ David Hoffman     │ 355.24 │ 2026-05-11 │
│ ORD00032 │ Mr. Philip Cannon │ 122.23 │ 2026-07-08 │
│ ORD00043 │ Maria Brown       │  54.55 │ 2026-07-02 │
│ ORD00130 │ Sandra Sanchez    │ 330.45 │ 2026-05-05 │
│ ORD00140 │ John Kelly        │ 145.57 │ 2026-07-19 │
│ ORD00191 │ Jordan Morris     │ 278.49 │ 2026-07-11 │
│ ORD00201 │ Michael Mitchell  │ 107.57 │ 2026-05-04 │
│ ORD00244 │ Jerome Cox        │ 286.55 │ 2026-07-09 │
│    ·     │     ·             │    ·   │     ·      │
│    ·     │     ·             │    ·   │     ·      │
│    ·     │     ·             │    ·   │     ·      │
│ ORD00756 │ Tiffany Curtis    │ 397.28 │ 2026-05-23 │
│ ORD00781

In [13]:
find_amount_discrepancies(con, "orders", "deliveries", "order_id", "amount", "delivered_amount").show()


┌──────────┬──────────┬──────────┬────────────────────┐
│ order_id │ amount_a │ amount_b │     difference     │
│ varchar  │  double  │  double  │       double       │
├──────────┼──────────┼──────────┼────────────────────┤
│ ORD00003 │   447.17 │    425.5 │ 21.670000000000016 │
│ ORD00012 │   174.93 │   169.19 │  5.740000000000009 │
│ ORD00064 │   160.68 │   116.66 │  44.02000000000001 │
│ ORD00065 │   474.01 │    449.2 │ 24.810000000000002 │
│ ORD00080 │   431.22 │   418.56 │ 12.660000000000025 │
│ ORD00085 │    103.3 │    81.18 │  22.11999999999999 │
│ ORD00088 │   492.24 │   471.51 │ 20.730000000000018 │
│ ORD00108 │   376.51 │   346.89 │ 29.620000000000005 │
│ ORD00139 │   424.59 │   402.83 │  21.75999999999999 │
│ ORD00144 │    14.74 │   -29.53 │              44.27 │
│    ·     │      ·   │      ·   │                ·   │
│    ·     │      ·   │      ·   │                ·   │
│    ·     │      ·   │      ·   │                ·   │
│ ORD00797 │   131.48 │   116.26 │ 15.2199999999

In [14]:
find_duplicates(con, "deliveries", "order_id").show()

┌──────────┬──────────────┐
│ order_id │ record_count │
│ varchar  │    int64     │
├──────────┼──────────────┤
│ ORD00113 │            2 │
│ ORD00151 │            2 │
│ ORD00262 │            2 │
│ ORD00489 │            2 │
│ ORD00774 │            2 │
│ ORD00745 │            2 │
│ ORD00171 │            2 │
│ ORD00208 │            2 │
│ ORD00267 │            2 │
│ ORD00274 │            2 │
│    ·     │            · │
│    ·     │            · │
│    ·     │            · │
│ ORD00500 │            2 │
│ ORD00877 │            2 │
│ ORD00587 │            2 │
│ ORD00898 │            2 │
│ ORD00067 │            2 │
│ ORD00269 │            2 │
│ ORD00534 │            2 │
│ ORD00133 │            2 │
│ ORD00508 │            2 │
│ ORD00893 │            2 │
└──────────┴──────────────┘
  36 rows       2 columns
  (20 shown)              



In [15]:
import duckdb
con = duckdb.connect("./reconciliation.duckdb")
con.sql("SHOW TABLES").show()

┌───────────────────────────────┐
│             name              │
│            varchar            │
├───────────────────────────────┤
│ deliveries                    │
│ mart_amount_discrepancies     │
│ mart_duplicate_deliveries     │
│ mart_missing_deliveries       │
│ orders                        │
│ prep_orders_deliveries_merged │
│ stg_deliveries                │
│ stg_orders                    │
└───────────────────────────────┘



In [16]:
con.sql("SELECT * FROM mart_missing_deliveries").show()


┌──────────┬───────────────────┬──────────────┬────────────┬──────────────────┬───────────────┐
│ order_id │     customer      │ order_amount │ order_date │ delivered_amount │ delivery_date │
│ varchar  │      varchar      │    double    │    date    │      double      │     date      │
├──────────┼───────────────────┼──────────────┼────────────┼──────────────────┼───────────────┤
│ ORD00000 │ Patrick Sanchez   │       323.32 │ 2026-06-29 │             NULL │ NULL          │
│ ORD00005 │ Melissa Peterson  │       257.62 │ 2026-07-10 │             NULL │ NULL          │
│ ORD00019 │ David Hoffman     │       355.24 │ 2026-05-11 │             NULL │ NULL          │
│ ORD00032 │ Mr. Philip Cannon │       122.23 │ 2026-07-08 │             NULL │ NULL          │
│ ORD00043 │ Maria Brown       │        54.55 │ 2026-07-02 │             NULL │ NULL          │
│ ORD00130 │ Sandra Sanchez    │       330.45 │ 2026-05-05 │             NULL │ NULL          │
│ ORD00140 │ John Kelly        │       1

In [17]:
con.sql("SELECT * FROM mart_amount_discrepancies").show()

┌──────────┬──────────────┬──────────────────┬────────────────────┐
│ order_id │ order_amount │ delivered_amount │     difference     │
│ varchar  │    double    │      double      │       double       │
├──────────┼──────────────┼──────────────────┼────────────────────┤
│ ORD00003 │       447.17 │            425.5 │ 21.670000000000016 │
│ ORD00012 │       174.93 │           169.19 │  5.740000000000009 │
│ ORD00064 │       160.68 │           116.66 │  44.02000000000001 │
│ ORD00065 │       474.01 │            449.2 │ 24.810000000000002 │
│ ORD00080 │       431.22 │           418.56 │ 12.660000000000025 │
│ ORD00085 │        103.3 │            81.18 │  22.11999999999999 │
│ ORD00088 │       492.24 │           471.51 │ 20.730000000000018 │
│ ORD00108 │       376.51 │           346.89 │ 29.620000000000005 │
│ ORD00139 │       424.59 │           402.83 │  21.75999999999999 │
│ ORD00144 │        14.74 │           -29.53 │              44.27 │
│    ·     │          ·   │              ·   │  

In [18]:
con.sql("SELECT * FROM mart_duplicate_deliveries").show()

┌──────────┬────────────────┐
│ order_id │ delivery_count │
│ varchar  │     int64      │
├──────────┼────────────────┤
│ ORD00262 │              2 │
│ ORD00489 │              2 │
│ ORD00774 │              2 │
│ ORD00171 │              2 │
│ ORD00208 │              2 │
│ ORD00267 │              2 │
│ ORD00274 │              2 │
│ ORD00161 │              2 │
│ ORD00372 │              2 │
│ ORD00683 │              2 │
│    ·     │              · │
│    ·     │              · │
│    ·     │              · │
│ ORD00151 │              2 │
│ ORD00745 │              2 │
│ ORD00133 │              2 │
│ ORD00508 │              2 │
│ ORD00893 │              2 │
│ ORD00067 │              2 │
│ ORD00269 │              2 │
│ ORD00534 │              2 │
│ ORD00587 │              2 │
│ ORD00898 │              2 │
└──────────┴────────────────┘
  36 rows         2 columns
  (20 shown)                

